# Generated-response evaluation

This notebook summarizes final evaluations that operate on model responses rather than preference-pair margins. Run it only after method selection is frozen and the evaluation jobs in `docs/EVALUATION.md` are complete.

- **Safety harm rate:** lower is better.
- **XSTest over-refusal:** lower is better on safe prompts; unsafe refusal is reported separately.
- **IFEval:** higher means instructions were followed more often.
- **Pairwise judge score:** above 50% favors the candidate, with ties worth half a win.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ARTIFACT_DIR = Path(os.environ['MFR_OUTPUT_DIR']).expanduser().resolve()
RUNS_DIR = ARTIFACT_DIR / 'runs'
print('Reading evaluations from', RUNS_DIR)

## 1. Safety and XSTest
Every completed WildGuard summary is shown together. Missing rows mean that evaluation was not run for that model.

In [ ]:
rows = []
for path in sorted(RUNS_DIR.glob('*/generation/final_eval/safety/summary.csv')):
    run_dir = path.parents[3]
    settings = json.loads((run_dir / 'settings.json').read_text())
    frame = pd.read_csv(path)
    frame.insert(0, 'run_name', settings['run_name'])
    frame.insert(1, 'method', settings['method'])
    frame.insert(2, 'order', settings['order_id'])
    frame.insert(3, 'seed', settings['seed'])
    rows.append(frame)
safety = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
display(safety.round(2)) if len(safety) else print('No safety evaluations found yet.')

In [ ]:
if len(safety):
    metrics = ['harm_rate_pct', 'over_refusal_rate_pct', 'unsafe_refusal_rate_pct']
    available = [m for m in metrics if m in safety]
    plot_data = safety.groupby(['method', 'suite'], observed=True)[available].mean().reset_index()
    display(plot_data.round(2))
    for metric in available:
        plot_data.pivot(index='method', columns='suite', values=metric).plot.bar(figsize=(9, 4))
        plt.ylabel(metric.replace('_', ' ').replace(' pct', ' (%)'))
        plt.title(metric.replace('_', ' ').title())
        plt.xticks(rotation=35, ha='right')
        plt.tight_layout(); plt.show()

## 2. IFEval
IFEval has strict and loose prompt-level and instruction-level scores. The code below reads the official `lm-eval` result files without assuming a job ID.

In [ ]:
ifeval_rows = []
for run_dir in sorted(RUNS_DIR.glob('*')):
    folder = run_dir / 'generation' / 'final_eval' / 'ifeval'
    if not folder.exists(): continue
    settings = json.loads((run_dir / 'settings.json').read_text())
    for path in folder.rglob('*.json'):
        try: payload = json.loads(path.read_text())
        except Exception: continue
        result = payload.get('results', {}).get('ifeval')
        if not isinstance(result, dict): continue
        row = {'run_name': settings['run_name'], 'method': settings['method'], 'order': settings['order_id'], 'seed': settings['seed']}
        row.update({k: v for k, v in result.items() if isinstance(v, (int, float))})
        ifeval_rows.append(row); break
ifeval = pd.DataFrame(ifeval_rows)
display(ifeval.round(3)) if len(ifeval) else print('No IFEval results found yet.')

## 3. Helpfulness and quality judge comparisons
Each answer is judged twice with candidate/baseline positions reversed. A disagreement becomes a tie and position consistency is reported.

In [ ]:
judge_rows = []
for path in sorted(RUNS_DIR.glob('*/generation/final_eval/judges/*/summary.csv')):
    frame = pd.read_csv(path)
    frame.insert(0, 'comparison_folder', path.parent.name)
    judge_rows.append(frame)
judges = pd.concat(judge_rows, ignore_index=True) if judge_rows else pd.DataFrame()
display(judges.round(2)) if len(judges) else print('No pairwise judge results found yet.')
if len(judges):
    judges.groupby(['candidate', 'baseline', 'behavior'])['candidate_score_pct'].mean().unstack().plot.bar(figsize=(9, 4), ylim=(0, 100))
    plt.axhline(50, color='black', linewidth=1, linestyle='--')
    plt.ylabel('Candidate score (%)'); plt.title('Position-controlled pairwise judge')
    plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()